In [ ]:
"""
Diagnostyka: czy obliczenia w bezsloncares_pipeline.py mają sens fizyczny?

Uruchom PO wygenerowaniu OuluResiduumBezSlonca.csv (czyli po odpaleniu
bezsloncares_pipeline.py). Ten skrypt niczego nie liczy od nowa poza
kilkoma tanimi kontrolami -- tylko sprawdza, czy wyniki wyglądają sensownie.

Uruchomienie: umieść ten plik w tym samym folderze co bezsloncares_pipeline.py
i odpal: python sprawdz_fizyke.py
"""

from pathlib import Path
import numpy as np
import pandas as pd

from bezslonca import (
    yield_1000, yield_function, J_modulated, N_star,
    C_alpha, OULU_RC, OULU_H, OUTPUT_FILE,
)

print("=" * 70)
print("TEST 1: Czy Y(1000,E) zgadza się (w granicach kilku %) z Tabelą 1 Misheva?")
print("=" * 70)
# Wartości referencyjne wprost z Tabeli 1 artykułu Misheva (proton, h=1000 g/cm^2)
tabela_1_mishev = {
    1.27: 1.21e-3,
    9.11: 1.09e-1,
    99.1: 9.92e-1,
    999.1: 6.67e+0,
}
for E, oczekiwane in tabela_1_mishev.items():
    policzone = yield_1000(E, "proton")
    blad_pct = 100 * abs(policzone - oczekiwane) / oczekiwane
    status = "OK" if blad_pct < 15 else "PODEJRZANE"
    print(f"  E={E:>7.2f} GeV: policzone={policzone:.4e}, "
          f"tabela={oczekiwane:.4e}, błąd={blad_pct:.1f}% [{status}]")
print("  (Artykuł deklaruje dokładność parametryzacji rzędu kilku % -- błędy")
print("   powyżej ~15% sugerowałyby pomyłkę w przepisaniu współczynników.)\n")


print("=" * 70)
print("TEST 2: Czy Y(h,E) na głębokości Oulu jest NIŻSZE niż na 1000 g/cm^2?")
print("=" * 70)
for E in [1.27, 9.11, 99.1]:
    y1000 = yield_1000(E, "proton")
    y_oulu = yield_function(E, OULU_H, "proton")
    stosunek = y_oulu / y1000
    status = "OK" if stosunek < 1.0 else "PODEJRZANE (powinno być < 1)"
    print(f"  E={E:>7.2f} GeV: Y(1000)={y1000:.4e}, Y({OULU_H})={y_oulu:.4e}, "
          f"stosunek={stosunek:.4f} [{status}]")
print()


print("=" * 70)
print("TEST 3: Czy J_modulated MALEJE z rosnącym phi? (silniejsza modulacja")
print("        słoneczna = mniej cząstek dociera do Ziemi)")
print("=" * 70)
wartosci = [J_modulated(1.0, phi, "proton") for phi in [300, 500, 700, 1000, 1500]]
malejace = all(wartosci[i] > wartosci[i+1] for i in range(len(wartosci)-1))
print(f"  J(T=1 GeV) dla phi=300,500,700,1000,1500: "
      f"{[f'{w:.1f}' for w in wartosci]}")
print(f"  Monotonicznie malejące: {'OK' if malejace else 'PODEJRZANE -- BŁĄD'}\n")


print("=" * 70)
print("TEST 4: Czy N*(phi) dla Oulu MALEJE monotonicznie w całym zakresie?")
print("=" * 70)
phi_test = np.arange(300, 2100, 50)
n_star_test = [N_star(phi, OULU_RC, OULU_H) for phi in phi_test]
malejace = all(n_star_test[i] > n_star_test[i+1] for i in range(len(n_star_test)-1))
print(f"  Sprawdzono {len(phi_test)} punktów od phi={phi_test[0]} do {phi_test[-1]} MV.")
print(f"  Monotonicznie malejące: {'OK' if malejace else 'PODEJRZANE -- BŁĄD'}\n")


print("=" * 70)
print("TEST 5: Czy C_alpha jest w rozsądnym zakresie (~0.3-0.4)?")
print("=" * 70)
for phi in [300, 500, 700, 1000, 1500]:
    c = C_alpha(phi)
    status = "OK" if 0.2 < c < 0.5 else "PODEJRZANE"
    print(f"  phi={phi:>5} MV -> C_alpha={c:.4f} [{status}]")
print()


# ============================================================
# Testy na WYNIKOWYM residuum (wymaga wcześniej uruchomionego pipeline'u)
# ============================================================
if not OUTPUT_FILE.exists():
    print(f"UWAGA: nie znaleziono {OUTPUT_FILE} -- uruchom najpierw "
          "bezsloncares_pipeline.py, żeby wykonać testy 6-8.")
else:
    df = pd.read_csv(OUTPUT_FILE)
    df["Data"] = pd.to_datetime(df["Data"])

    print("=" * 70)
    print("TEST 6: Czy kappa jest stabilna, zgodnie z artykułem (Oulu: ~0.78-1%")
    print("        względnego odchylenia standardowego)?")
    print("=" * 70)
    kappa_mean = df["kappa"].mean()
    kappa_std = df["kappa"].std()
    rel_std = 100 * kappa_std / kappa_mean
    status = "OK" if rel_std < 3 else "PODEJRZANE (oczekiwano rzędu 1%)"
    print(f"  kappa: średnia={kappa_mean:.6f}, względne std={rel_std:.2f}% [{status}]\n")

    print("=" * 70)
    print("TEST 7: Autokorelacja residuum -- czy roczna/27-dniowa struktura zniknęła?")
    print("=" * 70)
    res = df["Residuum"].to_numpy()

    def acf(x, lag):
        return np.corrcoef(x[:-lag], x[lag:])[0, 1]

    for lag, opis in [(1, "1 dzień"), (27, "rotacja Bartelsa"),
                       (182, "pół roku"), (365, "rok")]:
        wartosc = acf(res, lag)
        status = "OK (blisko 0)" if abs(wartosc) < 0.3 else "WCIĄŻ WYSOKA -- do sprawdzenia"
        print(f"  ACF(lag={lag:>4}, {opis:<20}) = {wartosc:>7.3f} [{status}]")
    print("  (Wysoka wartość na lag=182/365 może wskazywać na resztkowy dryf")
    print("   -- rozważ dostrojenie KAPPA_WINDOW_DAYS w głównym skrypcie.)\n")

    print("=" * 70)
    print("TEST 8: Czy resztkowy wieloletni dryf zniknął? (średnie roczne residuum)")
    print("=" * 70)
    df["Rok"] = df["Data"].dt.year
    roczne = df.groupby("Rok")["Residuum"].mean()
    print(f"  Zakres średnich rocznych: {roczne.min():.2f} do {roczne.max():.2f}")
    print(f"  Odchylenie std średnich rocznych: {roczne.std():.2f}")
    print(f"  (Dla porównania: odch. std samego residuum = {df['Residuum'].std():.2f})")
    stosunek = roczne.std() / df["Residuum"].std()
    status = "OK" if stosunek < 0.3 else "PODEJRZANE -- możliwy resztkowy dryf"
    print(f"  Stosunek std(rocznych)/std(residuum) = {stosunek:.2f} [{status}]\n")

    print("=" * 70)
    print("TEST 9: Czy znane spadki Forbusha (Tabela 2, Vaisanen 2023) dają")
    print("        ujemne odchylenie w residuum?")
    print("=" * 70)
    df_idx = df.set_index("Data")
    spadki_forbusha = [
        ("1978-02-15", 8.12), ("1978-05-02", 6.96), ("1982-07-14", 12.98),
        ("1991-03-24", 8.66), ("1992-02-27", 6.28), ("2001-04-12", 8.09),
        ("2005-09-11", 8.67), ("2012-03-09", 6.44),
    ]
    for data_str, sila in spadki_forbusha:
        data = pd.Timestamp(data_str)
        if data in df_idx.index:
            okno = df_idx.loc[data - pd.Timedelta(days=1): data + pd.Timedelta(days=3), "Residuum"]
            minimum = okno.min()
            print(f"  {data_str} (siła {sila}%): min. residuum w oknie = {minimum:>8.1f}")
        else:
            print(f"  {data_str}: poza zakresem danych")
    print("  (Duże ujemne wartości są mile widziane, ale NIE są konieczne --")
    print("   phi samo w sobie już częściowo 'wyjaśnia' spadki Forbusha, patrz")
    print("   wcześniejsza dyskusja w rozmowie.)\n")

print("=" * 70)
print("KONIEC DIAGNOSTYKI")
print("=" * 70)